In [ ]:
import socket
import select
from Crypto.PublicKey import RSA
from Crypto.Random import get_random_bytes
from Crypto.Cipher import AES, PKCS1_OAEP

IP = "127.0.0.1"
PORT = 1234

private_key = RSA.import_key(open("private.pem").read())


server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server_socket.bind((IP, PORT))
server_socket.listen()
sockets_list = [server_socket]
clients = {}
session_keys = {}
print(f'Listening for connections on {IP}:{PORT}...')

#funkcija za dešifriranje  sporočil s session_key
def receive_message(client_socket, session_key):
    nonce = client_socket.recv(16)
    tag = client_socket.recv(16)
    ciphertext = client_socket.recv(1000)

    cipher_aes = AES.new(session_key, AES.MODE_EAX, nonce)
    data = cipher_aes.decrypt_and_verify(ciphertext, tag)
    return data

while True:
    read_sockets, _, _ = select.select(sockets_list, [], sockets_list) # Blokira, dokler nečesa ne sprejme
    for notified_socket in read_sockets:
        if notified_socket == server_socket:
            client_socket, client_address = server_socket.accept()
            
            # Najprej dobimo enc_session_key
            enc_session_key = client_socket.recv(private_key.size_in_bytes())

            # Ga dekriptiramo
            cipher_rsa = PKCS1_OAEP.new(private_key)
            session_key = cipher_rsa.decrypt(enc_session_key)
            
            # Ga shranimo v session_keys
            session_keys[client_socket] = session_key
            print('Session key is ', session_key)
            
            sockets_list.append(client_socket)
            user = receive_message(client_socket, session_keys[client_socket])
            if user is False:
                continue
            clients[client_socket] = user
            print('Sprejeta povezava od {}:{}, uporabniško ime: {}'.format(*client_address, user.decode('utf-8')))
        else:
            message = receive_message(notified_socket, session_keys[notified_socket])
            if message is False:
                print('Closed connection from: {}'.format(clients[notified_socket].decode('utf-8')))
                sockets_list.remove(notified_socket)
                del clients[notified_socket]
                continue
            user = clients[notified_socket]
            print(f'Prejeto sporočilo od {user.decode("utf-8")}: {message.decode("utf-8")}')

Listening for connections on 127.0.0.1:1234...
Session key is  b'\x97:\x8c}\xd4\xcc\x7f\xa4\xec\x1d\xe0M\xa40v\x99'
Sprejeta povezava od 127.0.0.1:43260, uporabniško ime: gogo
Prejeto sporočilo od gogo: hall
